In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
print("Iniciando a unificação de todas as tabelas do Home Credit...")

# 1. Tabela Principal
path = 'data/'
name_app = 'application_train.csv'
df_app = pd.read_csv(path + name_app, sep=',')

Iniciando a unificação de todas as tabelas do Home Credit...


In [5]:
df_app.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# ==========================================
# 2. BUREAU & BUREAU BALANCE
# ==========================================

name_bureau = 'bureau.csv'
name_bureaubal = 'bureau_balance.csv'

print("Processando bureau e bureau_balance...")
bureau = pd.read_csv(path + name_bureau)
bureaubal = pd.read_csv(path + name_bureaubal)

# Agrega o bureau_balance por empréstimo (SK_ID_BUREAU)
bb_agg = bureaubal.groupby('SK_ID_BUREAU').agg({
    'MONTHS_BALANCE': ['min', 'max', 'count'],
    'STATUS': lambda x: (x != '0').mean() # proporção de meses com atraso/status diferente de regular
})
bb_agg.columns = [f'BB_{col[0]}_{col[1]}'.upper() for col in bb_agg.columns]
bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')

# Agrega o bureau por cliente (SK_ID_CURR)
bureau['DEBT_DIFF'] = bureau['AMT_CREDIT_SUM'] - bureau['AMT_CREDIT_SUM_DEBT']
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    'CREDIT_DAY_OVERDUE': ['max', 'mean'],
    'AMT_CREDIT_SUM_DEBT': ['sum', 'mean'],
    'DEBT_DIFF': ['sum', 'mean']
})
bureau_agg.columns = [f'BUREAU_{col[0]}_{col[1]}'.upper() for col in bureau_agg.columns]
bureau_agg = bureau_agg.reset_index()

Processando bureau e bureau_balance...


In [9]:
# ==========================================
# 3. PREVIOUS APPLICATION & BALANCES (POS, CC, Inst)
# ==========================================

name_prev = 'previous_application.csv'
name_pos = 'POS_CASH_balance.csv'
name_cc = 'credit_card_balance.csv'
name_inst = 'installments_payments.csv'

print("Processando previous_application e saldos (POS, CC, Inst)...")
prev = pd.read_csv(path + name_prev)
pos = pd.read_csv(path + name_pos)
cc = pd.read_csv(path + name_cc)
inst = pd.read_csv(path + name_inst)

# POS_CASH agregado por SK_ID_PREV
pos_agg = pos.groupby('SK_ID_PREV').agg({
    'MONTHS_BALANCE': ['max', 'count'],
    'SK_DPD': ['max', 'mean'],
    'SK_DPD_DEF': ['max', 'mean']
})
pos_agg.columns = [f'POS_{col[0]}_{col[1]}'.upper() for col in pos_agg.columns]
prev = prev.merge(pos_agg, on='SK_ID_PREV', how='left')

# Credit Card agregado por SK_ID_PREV
cc_agg = cc.groupby('SK_ID_PREV').agg({
    'MONTHS_BALANCE': ['max', 'count'],
    'AMT_BALANCE': ['max', 'mean'],
    'SK_DPD': ['max', 'mean']
})
cc_agg.columns = [f'CC_{col[0]}_{col[1]}'.upper() for col in cc_agg.columns]
prev = prev.merge(cc_agg, on='SK_ID_PREV', how='left')

# Installments agregado por SK_ID_PREV
inst['PAYMENT_DIFF'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
inst['DAYS_DIFF'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst_agg = inst.groupby('SK_ID_PREV').agg({
    'NUM_INSTALMENT_VERSION': ['nunique'],
    'DAYS_DIFF': ['max', 'mean'],
    'PAYMENT_DIFF': ['sum', 'mean']
})
inst_agg.columns = [f'INST_{col[0]}_{col[1]}'.upper() for col in inst_agg.columns]
prev = prev.merge(inst_agg, on='SK_ID_PREV', how='left')

# Agrega previous_application final por cliente (SK_ID_CURR)
prev_agg = prev.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'AMT_ANNUITY': ['max', 'mean'],
    'AMT_CREDIT': ['max', 'mean'],
    'POS_SK_DPD_MAX': ['max', 'mean'],
    'CC_AMT_BALANCE_MEAN': ['mean'],
    'INST_DAYS_DIFF_MAX': ['max', 'mean']
})
prev_agg.columns = [f'PREV_{col[0]}_{col[1]}'.upper() for col in prev_agg.columns]
prev_agg = prev_agg.reset_index()


Processando previous_application e saldos (POS, CC, Inst)...


In [10]:

# ==========================================
# 4. MERGE FINAL NA BASE PRINCIPAL
# ==========================================

print("Consolidando a base mestre...")
master_df = df_app.merge(bureau_agg, on='SK_ID_CURR', how='left')
master_df = master_df.merge(prev_agg, on='SK_ID_CURR', how='left')

# Salvando resultado
output_file = 'home_credit_full_master.csv'
master_df.to_csv(output_file, index=False)
print(f"🎉 Base mestre completa salva em '{output_file}'!")
print(f"Formato final: {master_df.shape[0]:,} linhas e {master_df.shape[1]:,} colunas.")

Consolidando a base mestre...
🎉 Base mestre completa salva em 'home_credit_full_master.csv'!
Formato final: 307,511 linhas e 139 colunas.


In [11]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

name_base_mestre = 'home_credit_full_master.csv'

# 1. Carregar a base mestre unificada
print("A carregar a base mestre completa...")
df_master = pd.read_csv(path + name_base_mestre, sep=',')

# 2. Separar Target e Features
target_col = 'TARGET'
colunas_remover = [target_col, 'SK_ID_CURR']
features = [col for col in df_master.columns if col not in colunas_remover]

X = df_master[features]
y = df_master[target_col]

# 3. Split de Dados (Treino, Validação e Teste)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

# Converter colunas object/string para category (exigência do LightGBM)
for col in X_train.select_dtypes(include=['object', 'string']).columns:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 4. Treinamento do LightGBM
print("A treinar o LightGBM com todas as tabelas agregadas...")
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': 6,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

model = lgb.LGBMClassifier(**params, n_estimators=1000)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

# 5. Avaliação (AUC, Gini, KS)
def calcular_gini(y_true, y_pred_proba):
    auc = roc_auc_score(y_true, y_pred_proba)
    return 2 * auc - 1

def calcular_ks(y_true, y_pred_proba):
    df_temp = pd.DataFrame({'y_true': y_true, 'prob': y_pred_proba})
    df_temp = df_temp.sort_values(by='prob', ascending=False).reset_index(drop=True)
    df_temp['cum_bad'] = (df_temp['y_true'] == 1).cumsum() / (df_temp['y_true'] == 1).sum()
    df_temp['cum_good'] = (df_temp['y_true'] == 0).cumsum() / (df_temp['y_true'] == 0).sum()
    return np.max(np.abs(df_temp['cum_bad'] - df_temp['cum_good']))

preds_test = model.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, preds_test)
gini_test = calcular_gini(y_test, preds_test)
ks_test = calcular_ks(y_test, preds_test)

print("\n" + "="*50)
print("🚀 RESULTADOS DO MODELO COM A BASE MESTRE COMPLETA")
print("="*50)
print(f"Teste -> AUC: {auc_test:.4f} | Gini: {gini_test:.4f} | KS: {ks_test:.4f}")
print("="*50)

: 